# 04 — Fine-tuning direto em GGUF com llama.cpp (EXPERIMENTAL)

Este caderno existe para fins de treinamento/engenharia. O suporte de treinamento do `llama.cpp` é descrito como **WIP** e, no estado atual, o fluxo documentado trabalha com modelos **F32**, não com um Q4_K_M já quantizado.

**Não use este caminho como pipeline principal.** Para Qwen3, o caderno 02 (PEFT/QLoRA) + caderno 03 (GGUF) é muito mais previsível.

Também não assuma que toda arquitetura suportada para inferência é automaticamente suportada pelo `llama-finetune`; por isso o notebook começa fazendo checagens e deixa a execução de treino explicitamente opt-in.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json

LLAMA_DIR = Path('tools/llama.cpp')
BUILD = LLAMA_DIR / 'build'
MODEL_HF = Path('artifacts/qwen3-0.6b-raciocinio-merged')
EXP_DIR = Path('artifacts/gguf_finetune_experimental')
EXP_DIR.mkdir(parents=True, exist_ok=True)

## 1. Verificar se `llama-finetune` foi compilado

In [ ]:
def find_binary(name):
    for p in [BUILD/'bin'/name, BUILD/'bin'/f'{name}.exe', BUILD/name, BUILD/f'{name}.exe']:
        if p.exists(): return p
    return None

finetune = find_binary('llama-finetune')
print('llama-finetune:', finetune)

if finetune is None:
    print('Recompile uma versão recente do llama.cpp. O target de training pode variar conforme a revisão.')
else:
    subprocess.run([str(finetune), '--help'], check=False)

## 2. Preparar um GGUF F32

O exemplo oficial de treinamento usa F32. Isso é muito maior que Q4/Q8 e o treino exige memória adicional para gradientes/otimizador.

In [ ]:
F32 = EXP_DIR / 'qwen3-0.6b-raciocinio-f32.gguf'
converter = LLAMA_DIR / 'convert_hf_to_gguf.py'

if not F32.exists():
    subprocess.run([
        sys.executable, str(converter), str(MODEL_HF),
        '--outfile', str(F32), '--outtype', 'f32'
    ], check=True)
print(F32, f'{F32.stat().st_size/1024**3:.2f} GiB')

## 3. Criar corpus raw-text para o experimento

O `llama-finetune` documentado recebe `--file` com texto. Este formato é diferente do SFT conversacional estruturado do notebook 02.

In [ ]:
TRAIN_TXT = EXP_DIR / 'train.txt'
blocos = [
    'Pergunta: Quanto é 6 vezes 8?\nResposta: 48.',
    'Pergunta: Quanto é 25% de 120?\nResposta: 30.',
    'Pergunta: Qual a complexidade média de busca em um set hash?\nResposta: O(1).',
]
TRAIN_TXT.write_text(("\n\n".join(blocos) + "\n\n") * 100, encoding='utf-8')
print(TRAIN_TXT, TRAIN_TXT.stat().st_size, 'bytes')

## 4. Comando de prova de conceito

Por segurança didática, `RUN_EXPERIMENT` começa como `False`. Ative somente depois de confirmar que a revisão do `llama.cpp` usada no laboratório aceita a arquitetura Qwen3 no trainer.

Se a ferramenta rejeitar a arquitetura, isso **não significa** que o GGUF esteja inválido; significa apenas que o caminho de treinamento ainda não cobre aquele modelo.

In [ ]:
RUN_EXPERIMENT = False

if finetune and RUN_EXPERIMENT:
    cmd = [
        str(finetune),
        '--file', str(TRAIN_TXT),
        '--model', str(F32),
        '-c', '256',
        '-b', '64',
        '-ub', '64',
        '-ngl', '999' if os.environ.get('CUDA_VISIBLE_DEVICES', '') != '' else '0',
    ]
    print('Executando:', ' '.join(cmd))
    subprocess.run(cmd, check=False)
else:
    print('Experimento não executado. Defina RUN_EXPERIMENT=True após validar suporte/hardware.')

## Conclusão desta trilha

O valor deste notebook é mostrar por que “treinar um GGUF” e “usar GGUF no deploy” são problemas diferentes:

- GGUF é excelente como artefato de execução local;
- treino quantizado direto não é o fluxo mais maduro;
- PEFT/QLoRA em pesos HF permite tooling, datasets e métricas melhores;
- ao final, você converte apenas o modelo aprovado para GGUF.